# 2B development ablations at 1000 steps

Repaired-placement 1000-step location-conditioning results, matched shuffled-coordinate controls and the RGB-only modality ablation on the fixed BigEarthNet.txt `bench` population.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs/evaluation"
finetuning_root = repo_root / "outputs/finetuning"
pd.options.display.max_columns = None

primary_manifest = pd.DataFrame([
    {"evaluation_job": "11401", "training_job": "11383", "label": "no_loc", "condition": "no_loc", "location_format": "none", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11402", "training_job": "11384", "label": "loc_text 2dp", "condition": "loc_text", "location_format": "2 decimals", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11409", "training_job": "11400", "label": "loc_text integer", "condition": "loc_text", "location_format": "integer", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11403", "training_job": "11385", "label": "L10 8t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 8, "projection_lr_x": 5},
    {"evaluation_job": "11404", "training_job": "11387", "label": "L10 4t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 4, "projection_lr_x": 5},
    {"evaluation_job": "11405", "training_job": "11388", "label": "L10 8t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 8, "projection_lr_x": 2},
    {"evaluation_job": "11406", "training_job": "11389", "label": "L10 4t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 4, "projection_lr_x": 2},
    {"evaluation_job": "11410", "training_job": "11398", "label": "L40 8t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5},
    {"evaluation_job": "11411", "training_job": "11399", "label": "L40 4t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 4, "projection_lr_x": 2},
    {"evaluation_job": "11486", "training_job": "11485", "label": "loc_encoding all_visual", "condition": "loc_encoding", "location_format": "additive encoding", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11495", "training_job": "11494", "label": "loc_encoding S1/S2", "condition": "loc_encoding", "location_format": "additive encoding", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
])
primary_manifest["seed"] = 42
primary_manifest["family"] = primary_manifest["condition"].map({"no_loc": "Baseline", "loc_text": "Coordinate text", "loc_embed": "Location tokens", "loc_encoding": "Additive encoding"})
primary_manifest["representation"] = primary_manifest["location_format"]
primary_manifest["fusion_scope"] = "n/a"
primary_manifest.loc[primary_manifest["label"] == "loc_encoding all_visual", "fusion_scope"] = "RGB + S1/S2 tokens"
primary_manifest.loc[primary_manifest["label"] == "loc_encoding S1/S2", "fusion_scope"] = "S1/S2 tokens"
primary_manifest["projector"] = "n/a"
primary_manifest.loc[primary_manifest["condition"] == "loc_embed", "projector"] = "MLP"
additional_primary_manifest = pd.DataFrame([
    {"evaluation_job": "11555", "training_job": "11554", "label": "projected direct S1/S2", "condition": "loc_encoding", "family": "Additive encoding", "representation": "direct Fourier", "fusion_scope": "S1/S2 tokens", "projector": "linear", "location_format": "additive encoding", "satclip_l": None, "tokens": 0, "projection_lr_x": 5, "seed": 42},
    {"evaluation_job": "11558", "training_job": "11557", "label": "additive SatCLIP S1/S2", "condition": "loc_additive_satclip", "family": "Additive encoding", "representation": "SatCLIP L40", "fusion_scope": "S1/S2 tokens", "projector": "linear", "location_format": "additive SatCLIP", "satclip_l": 40, "tokens": 0, "projection_lr_x": 5, "seed": 42},
    {"evaluation_job": "11561", "training_job": "11560", "label": "L40 8t 5x, geolocation marker", "condition": "loc_embed", "family": "Location tokens", "representation": "SatCLIP L40", "fusion_scope": "extra tokens", "projector": "MLP", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "seed": 42},
    {"evaluation_job": "11593", "training_job": "11592", "label": "L40 8t 5x, compact", "condition": "loc_embed", "family": "Location tokens", "representation": "SatCLIP L40", "fusion_scope": "extra tokens", "projector": "linear", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "seed": 42},
    {"evaluation_job": "11596", "training_job": "11595", "label": "no_loc (seed 43)", "condition": "no_loc", "family": "Baseline", "representation": "none", "fusion_scope": "n/a", "projector": "n/a", "location_format": "none", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "seed": 43},
    {"evaluation_job": "11598", "training_job": "11597", "label": "loc_text integer (seed 43)", "condition": "loc_text", "family": "Coordinate text", "representation": "integer", "fusion_scope": "prompt text", "projector": "n/a", "location_format": "integer", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "seed": 43},
    {"evaluation_job": "11601", "training_job": "11600", "label": "L40 8t 5x (seed 43)", "condition": "loc_embed", "family": "Location tokens", "representation": "SatCLIP L40", "fusion_scope": "extra tokens", "projector": "MLP", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "seed": 43},
    {"evaluation_job": "11604", "training_job": "11603", "label": "L40 8t 5x, compact (seed 43)", "condition": "loc_embed", "family": "Location tokens", "representation": "SatCLIP L40", "fusion_scope": "extra tokens", "projector": "linear", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "seed": 43},
    {"evaluation_job": "11607", "training_job": "11606", "label": "additive SatCLIP S1/S2 (seed 43)", "condition": "loc_additive_satclip", "family": "Additive encoding", "representation": "SatCLIP L40", "fusion_scope": "S1/S2 tokens", "projector": "linear", "location_format": "additive SatCLIP", "satclip_l": 40, "tokens": 0, "projection_lr_x": 5, "seed": 43},
])
primary_manifest = pd.concat([primary_manifest, additional_primary_manifest], ignore_index=True)
primary_manifest["coordinate_setting"] = "correct"
primary_manifest["training_steps"] = 1000
counterfactual_manifest = pd.DataFrame([
    {"evaluation_job": "11420", "training_job": "11400", "label": "loc_text integer [shuffled]", "condition": "loc_text", "location_format": "integer", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "shuffled"},
    {"evaluation_job": "11421", "training_job": "11398", "label": "L40 8t 5x [shuffled]", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "coordinate_setting": "shuffled"},
    {"evaluation_job": "11487", "training_job": "11485", "label": "loc_encoding all_visual [shuffled]", "condition": "loc_encoding", "location_format": "additive encoding", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "shuffled"},
    {"evaluation_job": "11496", "training_job": "11494", "label": "loc_encoding S1/S2 [shuffled]", "condition": "loc_encoding", "location_format": "additive encoding", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "shuffled"},
])
additional_counterfactual_manifest = pd.DataFrame([
    {"evaluation_job": "11556", "training_job": "11554", "label": "projected direct S1/S2 [shuffled]", "condition": "loc_encoding", "coordinate_setting": "shuffled", "seed": 42},
    {"evaluation_job": "11559", "training_job": "11557", "label": "additive SatCLIP S1/S2 [shuffled]", "condition": "loc_additive_satclip", "coordinate_setting": "shuffled", "seed": 42},
    {"evaluation_job": "11562", "training_job": "11560", "label": "L40 8t 5x, geolocation marker [shuffled]", "condition": "loc_embed", "coordinate_setting": "shuffled", "seed": 42},
    {"evaluation_job": "11594", "training_job": "11592", "label": "L40 8t 5x, compact [shuffled]", "condition": "loc_embed", "coordinate_setting": "shuffled", "seed": 42},
    {"evaluation_job": "11599", "training_job": "11597", "label": "loc_text integer (seed 43) [shuffled]", "condition": "loc_text", "coordinate_setting": "shuffled", "seed": 43},
    {"evaluation_job": "11602", "training_job": "11600", "label": "L40 8t 5x (seed 43) [shuffled]", "condition": "loc_embed", "coordinate_setting": "shuffled", "seed": 43},
    {"evaluation_job": "11605", "training_job": "11603", "label": "L40 8t 5x, compact (seed 43) [shuffled]", "condition": "loc_embed", "coordinate_setting": "shuffled", "seed": 43},
    {"evaluation_job": "11608", "training_job": "11606", "label": "additive SatCLIP S1/S2 (seed 43) [shuffled]", "condition": "loc_additive_satclip", "coordinate_setting": "shuffled", "seed": 43},
])
counterfactual_manifest["seed"] = 42
counterfactual_manifest = pd.concat([counterfactual_manifest, additional_counterfactual_manifest], ignore_index=True)
counterfactual_manifest["training_steps"] = 1000
modality_manifest = pd.DataFrame([
    {"evaluation_job": "11491", "training_job": "11488", "label": "RGB only (100-step diagnostic)", "condition": "no_loc", "location_format": "none", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "not applicable", "training_steps": 100},
    {"evaluation_job": "11492", "training_job": "11489", "label": "RGB only", "condition": "no_loc", "location_format": "none", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "not applicable", "training_steps": 1000},
])
modality_manifest["seed"] = 42
modality_manifest["family"] = "Modality ablation"
modality_manifest["representation"] = "none"
modality_manifest["fusion_scope"] = "RGB only"
modality_manifest["projector"] = "n/a"
summary_manifest = pd.concat([primary_manifest, modality_manifest.query("training_steps == 1000")], ignore_index=True)
evaluation_inventory = pd.concat([primary_manifest, counterfactual_manifest, modality_manifest], ignore_index=True)
evaluation_inventory.set_index("evaluation_job")

In [ ]:
def scored_path(job, filename):
    return evaluation_root / str(job) / "scored_predictions" / filename

def artifacts_ready(job):
    return all(path.exists() for path in [
        evaluation_root / str(job) / "predictions.jsonl",
        scored_path(job, "summary.json"),
        scored_path(job, "sample_scores.jsonl"),
    ])

evaluation_inventory["available"] = evaluation_inventory["evaluation_job"].map(artifacts_ready)
pending_jobs = evaluation_inventory.loc[~evaluation_inventory["available"], "evaluation_job"].tolist()
expected_pending = {"11607", "11608"}
unexpected_missing = sorted(set(pending_jobs) - expected_pending)
if unexpected_missing:
    raise FileNotFoundError(f"Missing synced artifacts for unexpected jobs: {unexpected_missing}")
available_inventory = evaluation_inventory[evaluation_inventory["available"]].copy()
available_summary_manifest = summary_manifest[summary_manifest["evaluation_job"].isin(available_inventory["evaluation_job"])].copy()

all_summaries = {
    job: json.loads(scored_path(job, "summary.json").read_text(encoding="utf-8"))
    for job in available_inventory["evaluation_job"]
}
summaries = {job: all_summaries[job] for job in available_summary_manifest["evaluation_job"]}
print(f"Loaded {len(all_summaries)} scored evaluation runs. Pending sync: {pending_jobs or 'none'}.")

## Population integrity

In [ ]:
def load_sample_scores(job):
    return pd.read_json(scored_path(job, "sample_scores.jsonl"), lines=True)

sample_scores = {job: load_sample_scores(job) for job in available_inventory["evaluation_job"]}
reference_job = available_summary_manifest.iloc[0]["evaluation_job"]
reference = sample_scores[reference_job].sort_values("sample_id").reset_index(drop=True)
integrity_rows = []
for job, frame in sample_scores.items():
    ordered = frame.sort_values("sample_id").reset_index(drop=True)
    same_ids = ordered["sample_id"].equals(reference["sample_id"])
    same_metadata = all(
        ordered[column].equals(reference[column])
        for column in ["patch_id", "task_type", "task_category", "split"]
    )
    integrity_rows.append({
        "evaluation_job": job,
        "rows": len(frame),
        "unique_sample_ids": frame["sample_id"].nunique(),
        "same_ids": same_ids,
        "same_metadata": same_metadata,
    })
integrity = available_inventory[["evaluation_job", "label", "coordinate_setting"]].merge(pd.DataFrame(integrity_rows), on="evaluation_job")
assert integrity["same_ids"].all() and integrity["same_metadata"].all()
integrity

## Comprehensive 1000-step comparison

Correct-coordinate evaluations are shown here. Shuffled-coordinate controls are kept in their own section because they answer a different question. `Rank` is descriptive and averages ranks over BLEU-4, binary accuracy, MCQ accuracy and bbox mIoU.

In [ ]:
def headline_metrics(summary):
    by_type = {row["task_type"]: row for row in summary["by_task_type"]}
    return {
        "BLEU-4": summary["captioning"]["bleu4"],
        "METEOR": summary["captioning"]["meteor"],
        "CIDEr": summary["captioning"]["cider"],
        "Binary accuracy": by_type["binary"]["accuracy"],
        "MCQ accuracy": by_type["mcq"]["accuracy"],
        "BBox mIoU": by_type["bounding box"]["miou"],
    }

headline_columns = ["evaluation_job", "label", "seed", "family", "representation", "fusion_scope", "projector", "condition"]
headline = available_summary_manifest[headline_columns].copy()
headline = headline.join(pd.DataFrame([headline_metrics(summaries[job]) for job in headline["evaluation_job"]]))
primary = ["BLEU-4", "Binary accuracy", "MCQ accuracy", "BBox mIoU"]
metric_columns = list(headline_metrics(next(iter(summaries.values()))))
headline["Rank"] = headline[primary].rank(ascending=False).mean(axis=1)
headline_format = {metric: "{:.4f}" for metric in metric_columns}
headline_format["Rank"] = "{:.2f}"
display(
    headline.set_index("label")
    .drop(columns=["evaluation_job", "condition"])
    .style.format(headline_format)
    .highlight_max(subset=metric_columns, props="font-weight: bold")
    .highlight_min(subset=["Rank"], props="font-weight: bold")
)

headline_core_labels = ["no_loc", "loc_text integer", "L40 8t 5x", "additive SatCLIP S1/S2"]
display(
    headline.set_index("label").loc[headline_core_labels, ["family", *metric_columns, "Rank"]]
    .style.format(headline_format)
    .highlight_max(subset=metric_columns, props="font-weight: bold")
    .highlight_min(subset=["Rank"], props="font-weight: bold")
    .set_caption("Seed-42 headline: baseline, text, location tokens and strongest additive encoding")
)

In [ ]:
baseline_by_seed = (
    headline[headline["condition"] == "no_loc"]
    .drop_duplicates("seed")
    .set_index("seed")[primary]
)
headline_deltas = headline.set_index("label")[["seed", *primary]].copy()
for label, row in headline_deltas.iterrows():
    headline_deltas.loc[label, primary] = row[primary] - baseline_by_seed.loc[row["seed"]]
headline_deltas.style.format({"seed": "{:.0f}", **{metric: "{:+.4f}" for metric in primary}}).background_gradient(cmap="RdYlGn", axis=None, subset=primary, vmin=-0.03, vmax=0.03).set_caption("Delta versus no_loc at the same seed")

## Task-category detail and direct-geography MCQ

In [ ]:
category_rows = []
available_primary_manifest = primary_manifest[primary_manifest["evaluation_job"].isin(summaries)].copy()
for row in available_primary_manifest.itertuples(index=False):
    for score in summaries[row.evaluation_job]["by_task_category"]:
        category_rows.append({"label": row.label, "evaluation_job": row.evaluation_job, **score})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    table = (
        category_scores[category_scores["task_type"] == task_type]
        .pivot(index="label", columns="task_category", values=metric)
        .reindex(available_primary_manifest["label"])
    )
    table.index.name = None
    table.columns.name = None
    return table

mcq = category_table("mcq", "accuracy")
display(mcq.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy by category"))
display(mcq[["climate zone", "country", "season"]].style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Direct-geography MCQ accuracy"))

In [ ]:
display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy by category"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU by category"))

## Comparative research questions

Each row changes one principal design choice within a matched seed-42 comparison. The projected-direct comparison asks a concrete question: does a small learned linear adapter help the LLM make use of deterministic coordinate features? Because that variant also uses a compact feature basis and normalization, the result evaluates the complete projected-direct bridge rather than the linear layer in isolation.

In [ ]:
comparisons = [
    ("Integer minus 2dp text", "loc_text integer", "loc_text 2dp"),
    ("4t minus 8t at L10/5x", "L10 4t 5x", "L10 8t 5x"),
    ("4t minus 8t at L10/2x", "L10 4t 2x", "L10 8t 2x"),
    ("2x minus 5x at L10/8t", "L10 8t 2x", "L10 8t 5x"),
    ("2x minus 5x at L10/4t", "L10 4t 2x", "L10 4t 5x"),
    ("L40 minus L10 at 8t/5x", "L40 8t 5x", "L10 8t 5x"),
    ("L40 minus L10 at 4t/2x", "L40 4t 2x", "L10 4t 2x"),
    ("loc_encoding minus no_loc", "loc_encoding all_visual", "no_loc"),
    ("S1/S2 scope minus all-visual scope", "loc_encoding S1/S2", "loc_encoding all_visual"),
    ("Learned adapter minus fixed direct encoding", "projected direct S1/S2", "loc_encoding S1/S2"),
    ("SatCLIP features minus direct features, same additive bridge", "additive SatCLIP S1/S2", "projected direct S1/S2"),
    ("Additive SatCLIP minus no_loc", "additive SatCLIP S1/S2", "no_loc"),
    ("Geolocation marker minus coordinates marker", "L40 8t 5x, geolocation marker", "L40 8t 5x"),
    ("Compact minus MLP projector, seed 42", "L40 8t 5x, compact", "L40 8t 5x"),
    ("Compact minus MLP projector, seed 43", "L40 8t 5x, compact (seed 43)", "L40 8t 5x (seed 43)"),
]
headline_indexed = headline.set_index("label")
ablation_deltas = pd.DataFrame([
    {"Comparison": name, **(headline_indexed.loc[left, primary] - headline_indexed.loc[right, primary]).to_dict()}
    for name, left, right in comparisons
]).set_index("Comparison")
display(ablation_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.03, vmax=0.03))

direct_geography = ["climate zone", "country", "season"]
geography_deltas = pd.DataFrame([
    {"Comparison": name, **(mcq.loc[left, direct_geography] - mcq.loc[right, direct_geography]).to_dict()}
    for name, left, right in comparisons
]).set_index("Comparison")
geography_deltas.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.10, vmax=0.10)

## Selected full-run settings

In [ ]:
selection_pairs = [
    ("Coordinate text", "integer", "loc_text integer", "loc_text 2dp"),
    ("Location tokens", "8", "L10 8t 5x", "L10 4t 5x"),
    ("Projection LR", "5x", "L10 8t 5x", "L10 8t 2x"),
    ("SatCLIP L", "40", "L40 8t 5x", "L10 8t 5x"),
]
selection_rows = []
def win_tie_loss(delta, tolerance=1e-12):
    ties = delta.abs() <= tolerance
    wins = (delta > tolerance).sum()
    losses = (delta < -tolerance).sum()
    return f"{wins}/{ties.sum()}/{losses}"

for factor, selected, left, right in selection_pairs:
    primary_delta = headline_indexed.loc[left, primary] - headline_indexed.loc[right, primary]
    geography_delta = mcq.loc[left, direct_geography] - mcq.loc[right, direct_geography]
    selection_rows.append({
        "Factor": factor,
        "Selected": selected,
        "Primary W/T/L": win_tie_loss(primary_delta),
        "Direct geography W/T/L": win_tie_loss(geography_delta),
    })
pd.DataFrame(selection_rows).set_index("Factor")

## Robustness across training seeds

Seed 42 and seed 43 are separate training runs. Compare conditions against `no_loc` from the same seed; the range across two seeds is descriptive rather than a confidence interval.

In [ ]:
replication_labels = {
    "no_loc": {42: "no_loc", 43: "no_loc (seed 43)"},
    "loc_text integer": {42: "loc_text integer", 43: "loc_text integer (seed 43)"},
    "loc_embed L40 8t 5x": {42: "L40 8t 5x", 43: "L40 8t 5x (seed 43)"},
    "loc_embed compact": {42: "L40 8t 5x, compact", 43: "L40 8t 5x, compact (seed 43)"},
    "additive SatCLIP": {42: "additive SatCLIP S1/S2", 43: "additive SatCLIP S1/S2 (seed 43)"},
}
replication_rows = []
for variant, labels_by_seed in replication_labels.items():
    for seed, label in labels_by_seed.items():
        if label in headline_indexed.index:
            replication_rows.append({"Variant": variant, "Seed": seed, **headline_indexed.loc[label, metric_columns].to_dict()})
replication_scores = pd.DataFrame(replication_rows)
display(replication_scores.set_index(["Variant", "Seed"]).style.format({metric: "{:.4f}" for metric in metric_columns}))

seed_matched_rows = []
for _, row in replication_scores.iterrows():
    baseline_row = replication_scores[(replication_scores["Variant"] == "no_loc") & (replication_scores["Seed"] == row["Seed"])].iloc[0]
    seed_matched_rows.append({"Variant": row["Variant"], "Seed": row["Seed"], **{metric: row[metric] - baseline_row[metric] for metric in primary}})
seed_matched_deltas = pd.DataFrame(seed_matched_rows)
display(seed_matched_deltas.set_index(["Variant", "Seed"]).style.format("{:+.4f}").set_caption("Delta versus no_loc at the same seed"))

replicated = replication_scores.groupby("Variant").filter(lambda group: group["Seed"].nunique() == 2)
seed_ranges = replicated.groupby("Variant")[primary].agg(lambda values: values.max() - values.min())
display(seed_ranges.style.format("{:.4f}").set_caption("Absolute range across seeds 42 and 43"))

## Exact prediction disagreement

In [ ]:
def load_predictions(job):
    rows = []
    path = evaluation_root / job / "predictions.jsonl"
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            rows.append({key: row.get(key) for key in ["sample_id", "patch_id", "task_type", "task_category", "prediction"]})
    return pd.DataFrame(rows).set_index("sample_id").sort_index()

predictions = {row.label: load_predictions(row.evaluation_job) for row in available_primary_manifest.itertuples(index=False)}
disagreement_rows = []
for name, left, right in comparisons:
    left_frame, right_frame = predictions[left], predictions[right]
    changed = left_frame["prediction"] != right_frame["prediction"]
    disagreement_rows.append({"Comparison": name, "Task type": "all", "Rows": len(changed), "Changed": int(changed.sum()), "Changed fraction": changed.mean()})
    for task_type, indices in left_frame.groupby("task_type").groups.items():
        subset = changed.loc[indices]
        disagreement_rows.append({"Comparison": name, "Task type": task_type, "Rows": len(subset), "Changed": int(subset.sum()), "Changed fraction": subset.mean()})
disagreement = pd.DataFrame(disagreement_rows)
disagreement.pivot(index="Comparison", columns="Task type", values="Changed fraction").style.format("{:.1%}").set_caption("Exact generated-text disagreement")

## Does the model use the supplied coordinates?

Correct and shuffled evaluations are paired within the same trained adapter. Negative shuffled-minus-correct values mean that replacing the true coordinates hurt performance.

In [ ]:
counterfactual_pairs = pd.DataFrame([
    {"label": "loc_text integer", "correct_job": "11409", "shuffled_job": "11420"},
    {"label": "L40 8t 5x", "correct_job": "11410", "shuffled_job": "11421"},
    {"label": "loc_encoding all_visual", "correct_job": "11486", "shuffled_job": "11487"},
    {"label": "loc_encoding S1/S2", "correct_job": "11495", "shuffled_job": "11496"},
    {"label": "projected direct S1/S2", "correct_job": "11555", "shuffled_job": "11556"},
    {"label": "additive SatCLIP S1/S2", "correct_job": "11558", "shuffled_job": "11559"},
    {"label": "L40 8t 5x, geolocation marker", "correct_job": "11561", "shuffled_job": "11562"},
    {"label": "L40 8t 5x, compact", "correct_job": "11593", "shuffled_job": "11594"},
    {"label": "loc_text integer (seed 43)", "correct_job": "11598", "shuffled_job": "11599"},
    {"label": "L40 8t 5x (seed 43)", "correct_job": "11601", "shuffled_job": "11602"},
    {"label": "L40 8t 5x, compact (seed 43)", "correct_job": "11604", "shuffled_job": "11605"},
    {"label": "additive SatCLIP S1/S2 (seed 43)", "correct_job": "11607", "shuffled_job": "11608"},
])
counterfactual_pairs = counterfactual_pairs[
    counterfactual_pairs["correct_job"].isin(all_summaries) & counterfactual_pairs["shuffled_job"].isin(all_summaries)
].copy()
counterfactual_scores = []
counterfactual_deltas = []
counterfactual_disagreement = []
for row in counterfactual_pairs.itertuples(index=False):
    correct_metrics = pd.Series(headline_metrics(all_summaries[row.correct_job]))
    shuffled_metrics = pd.Series(headline_metrics(all_summaries[row.shuffled_job]))
    for setting, job, metrics in [
        ("correct", row.correct_job, correct_metrics),
        ("shuffled", row.shuffled_job, shuffled_metrics),
    ]:
        counterfactual_scores.append({"Condition": row.label, "Coordinate setting": setting, "Evaluation job": job, **metrics.to_dict()})
    counterfactual_deltas.append({"Condition": row.label, **(shuffled_metrics[primary] - correct_metrics[primary]).to_dict()})
    correct_predictions = load_predictions(row.correct_job)
    shuffled_predictions = load_predictions(row.shuffled_job)
    changed = correct_predictions["prediction"] != shuffled_predictions["prediction"]
    counterfactual_disagreement.append({"Condition": row.label, "Changed": int(changed.sum()), "Changed fraction": changed.mean()})

display(pd.DataFrame(counterfactual_scores).set_index(["Condition", "Coordinate setting"]).style.format({metric: "{:.4f}" for metric in metric_columns}).set_caption("Correct and shuffled evaluation scores"))
display(pd.DataFrame(counterfactual_deltas).set_index("Condition").style.format("{:+.4f}").set_caption("Shuffled minus correct"))
display(pd.DataFrame(counterfactual_disagreement).set_index("Condition").style.format({"Changed fraction": "{:.1%}"}))

direct_geography_rows = []
for row in counterfactual_pairs.itertuples(index=False):
    for setting, job in [("correct", row.correct_job), ("shuffled", row.shuffled_job)]:
        category_accuracy = {item["task_category"]: item["accuracy"] for item in all_summaries[job]["by_task_category"] if item["task_type"] == "mcq"}
        direct_geography_rows.append({"Condition": row.label, "Coordinate setting": setting, **{category: category_accuracy[category] for category in direct_geography}})
display(pd.DataFrame(direct_geography_rows).set_index(["Condition", "Coordinate setting"]).style.format("{:.3f}").set_caption("Direct-geography MCQ accuracy"))

In [ ]:
transition_rows = []
for row in counterfactual_pairs.itertuples(index=False):
    correct = sample_scores[row.correct_job][["sample_id", "task_type", "task_category", "correct"]].rename(columns={"correct": "correct_coordinates"})
    shuffled = sample_scores[row.shuffled_job][["sample_id", "correct"]].rename(columns={"correct": "shuffled_coordinates"})
    paired = correct.merge(shuffled, on="sample_id")
    paired = paired[paired["task_type"].isin(["binary", "mcq"])]
    for (task_type, task_category), group in paired.groupby(["task_type", "task_category"], sort=False):
        helpful = ((group["correct_coordinates"] == True) & (group["shuffled_coordinates"] == False)).sum()
        harmful = ((group["correct_coordinates"] == False) & (group["shuffled_coordinates"] == True)).sum()
        transition_rows.append({"Condition": row.label, "Task type": task_type, "Category": task_category, "Correct helps": helpful, "Correct harms": harmful, "Net correct": helpful - harmful})
transitions = pd.DataFrame(transition_rows)
display(transitions.set_index(["Condition", "Task type", "Category"]).style.background_gradient(subset=["Net correct"], cmap="RdYlGn", axis=None))

## RGB-only modality ablation

Published Qwen3-VL-8B is an unmatched zero-shot RGB reference, not part of the descriptive rank. Table 2 reports BLEU-4 0.57%; the detailed caption table reports 0.70%.

In [ ]:
published_reference = pd.DataFrame([
    {"Model": "Published Qwen3-VL-8B, zero-shot RGB", "BLEU-4": 0.0057, "Binary accuracy": 0.6196, "MCQ accuracy": 0.3755, "BBox mIoU": 0.1800},
    {"Model": "RGB only", **{metric: headline_indexed.loc["RGB only", metric] for metric in primary}},
]).set_index("Model")
display(published_reference.style.format("{:.4f}"))
(published_reference.loc["RGB only"] - published_reference.loc["Published Qwen3-VL-8B, zero-shot RGB"]).to_frame("Fine-tuned 2B minus published zero-shot 8B").style.format("{:+.4f}")

In [ ]:
rgb_rows = []
for row in modality_manifest.itertuples(index=False):
    rgb_rows.append({"label": row.label, "training_steps": row.training_steps, **headline_metrics(all_summaries[row.evaluation_job])})
rgb_scores = pd.DataFrame(rgb_rows).set_index("label")
display(rgb_scores.style.format({metric: "{:.4f}" for metric in metric_columns}))

rgb_1000 = pd.Series(headline_metrics(all_summaries["11492"]))
multisensor_1000 = pd.Series(headline_metrics(all_summaries["11401"]))
rgb_delta = (rgb_1000 - multisensor_1000).to_frame("RGB only minus RGB + S1/S2")
display(rgb_delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.03, vmax=0.03))

In [ ]:
def mcq_category_series(summary):
    return pd.Series({row["task_category"]: row["accuracy"] for row in summary["by_task_category"] if row["task_type"] == "mcq"})

rgb_geography_delta = (
    mcq_category_series(all_summaries["11492"])[direct_geography]
    - mcq_category_series(all_summaries["11401"])[direct_geography]
).to_frame("RGB only minus RGB + S1/S2")
display(rgb_geography_delta.style.format("{:+.3f}"))

rgb_predictions = load_predictions("11492")
multisensor_predictions = load_predictions("11401")
rgb_changed = rgb_predictions["prediction"] != multisensor_predictions["prediction"]
pd.Series({"Changed": int(rgb_changed.sum()), "Changed fraction": rgb_changed.mean()}).to_frame("RGB only versus RGB + S1/S2").T.style.format({"Changed fraction": "{:.1%}"})

## Validation trajectories (diagnostic only)

In [ ]:
trajectory_labels = [
    "no_loc", "loc_text integer", "L40 8t 5x", "loc_encoding all_visual", "loc_encoding S1/S2",
    "projected direct S1/S2", "additive SatCLIP S1/S2", "L40 8t 5x, compact",
    "no_loc (seed 43)", "loc_text integer (seed 43)", "L40 8t 5x (seed 43)",
    "L40 8t 5x, compact (seed 43)", "additive SatCLIP S1/S2 (seed 43)",
]
curve_manifest = summary_manifest[summary_manifest["label"].isin(trajectory_labels)].copy()
curve_rows = []
for row in curve_manifest.itertuples(index=False):
    event_files = sorted((finetuning_root / row.training_job / "lightning_logs").glob("version_*/events.out.tfevents.*"))
    if not event_files:
        raise FileNotFoundError(f"Missing TensorBoard event file for training job {row.training_job}")
    accumulator = EventAccumulator(str(event_files[-1]), size_guidance={"scalars": 0})
    accumulator.Reload()
    curve_rows.extend({"label": row.label, "seed": row.seed, "step": event.step + 1, "val_loss": event.value} for event in accumulator.Scalars("val/loss"))
curves = pd.DataFrame(curve_rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, seed in zip(axes, [42, 43]):
    for label, group in curves[curves["seed"] == seed].groupby("label", sort=False):
        ax.plot(group["step"], group["val_loss"], marker="o", linewidth=1.4, markersize=3, label=label)
    ax.set(xlabel="Optimizer step", title=f"Seed {seed}")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("Teacher-forced validation loss")
fig.suptitle("Selected repaired-placement 1000-step trajectories")
fig.tight_layout()
plt.show()

final_validation = (
    curves.sort_values(["label", "step"])
    .groupby("label", sort=False).tail(1)
    .set_index("label")[["step", "val_loss"]]
    .reindex(curve_manifest["label"])
)
final_validation.style.format({"step": "{:.0f}", "val_loss": "{:.6f}"}).highlight_min(subset=["val_loss"], props="font-weight: bold")

## Reading notes

- Metrics remain separate; rank is a descriptive summary of four primary metrics.
- The comprehensive table retains weaker and negative variants because they identify which representation and fusion decisions matter.
- Ablation deltas are matched within a seed. Seed-42 and seed-43 runs are never treated as directly paired.
- Two-seed ranges are descriptive and are not confidence intervals.
- Validation loss is diagnostic only.
- The 100-step RGB-only evaluation is an early-training diagnostic; the matched modality comparison uses the 1000-step result.
- Shuffled-coordinate results measure whether a trained adapter uses the supplied coordinates; they are not additional independently trained models.
- The `bench` results informed configuration selection. A later condensed notebook may extract the final comparisons after the development conclusions stabilize.